# Task 3: Qwen2.5-Coder-7B QLoRA Training & Public Test Submission (Kaggle Runner)

Notebook này dùng để huấn luyện **Qwen2.5-Coder-7B-Instruct** trên **2 tập dữ liệu (Real + Synthetic)** và chạy đánh giá, sinh file nộp bài cho **Task 3 (Sinh nhận xét sư phạm tiếng Việt)** trên tập **Public Test (328 mẫu)**.

### Hướng dẫn thiết lập trên Kaggle:
1. **Kaggle Settings** (cột bên phải):
   - **Accelerator**: **GPU T4 x2** hoặc **GPU P100** (bắt buộc để huấn luyện QLoRA và chạy model).
   - **Internet**: **On** (bắt buộc bật để tải repo, weights Qwen và package).
2. **Kaggle Datasets / Input** (Nút *+ Add Data* ở góc phải trên):
   - Upload file `generate-dataset.zip` (synthetic dataset) và `sample_dataset.zip` / `public_test_release.zip`.
   - Hoặc để notebook tự động tải thông qua Google Drive ID.
3. **Kết quả đầu ra**:
   - File nộp bài `task3_submission.json` và báo cáo `evaluation.json` sẽ nằm trong thư mục `/kaggle/working/` để tải về nộp trực tiếp lên Leaderboard.


In [ ]:
# ==============================================================================
# CẤU HÌNH THÍ NGHIỆM TASK 3 (TRAIN 2 TẬP DATA & EVAL PUBLIC TEST)
# ==============================================================================

# 1. Thông tin Git repo & nhánh cần làm việc
REPO = "github.com/tiendinhquang2104/llm-final-project.git"
BRANCH = "task-3"

# 2. Google Drive File ID (dùng khi chưa upload file lên Kaggle qua '+ Add Data')
# DRIVE_ID_PUBLIC_SAMPLE: file zip chứa sample_dataset.zip + public_test_release.zip
DRIVE_ID_PUBLIC_SAMPLE = "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj"
# DRIVE_ID_SYNTHETIC: file zip chứa generate-dataset (nếu có lưu trên Google Drive)
DRIVE_ID_SYNTHETIC = ""  # Điền ID Google Drive nếu có

# 3. Chế độ chạy (MODE):
# - "qwen_qlora": Huấn luyện Qwen-7B bằng QLoRA trên CẢ 2 TẬP DỮ LIỆU (mixed_balanced), sau đó sinh kết quả nộp bài
# - "qwen_baseline": Chạy Qwen-7B Zero-shot trực tiếp trên Public test (không huấn luyện)
# - "gemini": Sinh nhận xét bằng Gemini API (gemini-2.5-flash)
MODE = "qwen_qlora"

# 4. Tập dữ liệu đánh giá:
# - "public": Chạy qua toàn bộ 328 mẫu của Public Test và xuất file JSON nộp bài
# - "real-val": Đánh giá nội bộ trên 6 bài validation có nhãn gold
EVAL_SPLIT = "public"

# 5. Cấu hình huấn luyện QLoRA:
EXPERIMENT = "mixed_balanced"  # Cân bằng tối ưu giữa real data (26 bài) và synthetic data (496 bài)
EPOCHS = 3                     # Số epochs huấn luyện
MAX_STEPS = None               # Đặt số nguyên (ví dụ 100) nếu muốn test nhanh, hoặc None để train hết epochs

WORK = "/kaggle/tmp"


In [ ]:
import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

token = get_secret("GITHUB_TOKEN")
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

os.makedirs(WORK, exist_ok=True)
REPO_DIR = f"{WORK}/repo"

if not os.path.exists(REPO_DIR):
    print(f"Đang clone {REPO} (nhánh: {BRANCH})...")
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone thất bại — hãy kiểm tra lại REPO, BRANCH hoặc GITHUB_TOKEN") from None
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://{REPO}"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Nạp API Keys từ Kaggle Secrets vào môi trường và file .env
gemini_key = get_secret("GEMINI_API_KEY")
openai_key = get_secret("OPENAI_API_KEY")
hf_token = get_secret("HF_TOKEN")

env_lines = []
if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key
    env_lines.append(f"GEMINI_API_KEY={gemini_key}")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    env_lines.append(f"OPENAI_API_KEY={openai_key}")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    env_lines.append(f"HF_TOKEN={hf_token}")

if env_lines:
    with open(".env", "w", encoding="utf-8") as f:
        f.write("\n".join(env_lines) + "\n")

print("Thư mục làm việc hiện tại:", os.getcwd())
print("Commit mới nhất:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q gdown pyyaml pytest
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes peft trl datasets
if MODE in ("gemini", "openai"):
    !pip install -q google-genai openai

# Kiểm tra trạng thái GPU
!nvidia-smi


In [ ]:
import os, glob, shutil, gdown

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/synth", exist_ok=True)
os.makedirs(f"{WORK}/download", exist_ok=True)

# ------------------------------------------------------------------------------
# 1. BỘ DỮ LIỆU THẬT & PUBLIC TEST (sample_dataset + public_test_release)
# ------------------------------------------------------------------------------
print("=== 1. Chuẩn bị Real Data & Public Test ===")
found_sample = glob.glob("/kaggle/input/**/sample_dataset", recursive=True)
found_public = glob.glob("/kaggle/input/**/public_test_release", recursive=True)
found_real_zips = glob.glob("/kaggle/input/**/sample_dataset*.zip", recursive=True) or glob.glob("/kaggle/input/**/*cham-diem*.zip", recursive=True)

if found_sample and not os.path.exists("data/raw/sample_dataset"):
    shutil.copytree(found_sample[0], "data/raw/sample_dataset")
    print(f"-> Đã copy sample_dataset từ Kaggle Input: {found_sample[0]}")

if found_public and not os.path.exists("data/raw/public_test_release"):
    shutil.copytree(found_public[0], "data/raw/public_test_release")
    print(f"-> Đã copy public_test_release từ Kaggle Input: {found_public[0]}")

if not os.path.exists("data/raw/sample_dataset") or not os.path.exists("data/raw/public_test_release"):
    zip_source = None
    if found_real_zips:
        zip_source = found_real_zips[0]
        print(f"-> Dùng file zip từ Kaggle Input: {zip_source}")
    else:
        drive_id = DRIVE_ID_PUBLIC_SAMPLE or get_secret("PUBLIC_TEST_DRIVE_ID") or get_secret("DRIVE_ID")
        if drive_id:
            zip_dest = f"{WORK}/download/public_and_sample.zip"
            print(f"-> Đang tải từ Google Drive (ID: {drive_id})...")
            gdown.download(id=drive_id, output=zip_dest, quiet=False)
            zip_source = zip_dest

    if zip_source and os.path.exists(zip_source):
        !python scripts/unpack_data.py {os.path.dirname(zip_source)} --dest data/raw
        for sub_zip in glob.glob("data/raw/*.zip"):
            !unzip -q -o {sub_zip} -d data/raw

!rm -rf data/raw/__MACOSX

# ------------------------------------------------------------------------------
# 2. BỘ DỮ LIỆU TỔNG HỢP (generate-dataset)
# ------------------------------------------------------------------------------
print("\n=== 2. Chuẩn bị Synthetic Data (generate-dataset) ===")
synth_dest = "data/synth/generate-dataset"
found_synth_dirs = [d for d in glob.glob("/kaggle/input/**/generate-dataset*", recursive=True) if os.path.isdir(d) and os.path.exists(os.path.join(d, "task3_feedback.json"))]
found_synth_zips = glob.glob("/kaggle/input/**/generate-dataset*.zip", recursive=True)

if found_synth_dirs and not os.path.exists(synth_dest):
    shutil.copytree(found_synth_dirs[0], synth_dest)
    print(f"-> Đã copy generate-dataset từ Kaggle Input: {found_synth_dirs[0]}")
elif found_synth_zips and not os.path.exists(synth_dest):
    print(f"-> Giải nén generate-dataset.zip từ Kaggle Input: {found_synth_zips[0]}")
    !unzip -q -o {found_synth_zips[0]} -d data/synth
else:
    drive_synth_id = DRIVE_ID_SYNTHETIC or get_secret("SYNTHETIC_DRIVE_ID")
    if drive_synth_id:
        synth_zip = f"{WORK}/download/generate_dataset.zip"
        print(f"-> Đang tải generate-dataset từ Google Drive (ID: {drive_synth_id})...")
        gdown.download(id=drive_synth_id, output=synth_zip, quiet=False)
        !unzip -q -o {synth_zip} -d data/synth
    elif not os.path.exists(synth_dest):
        print("⚠️ LƯU Ý: Chưa thấy thư mục generate-dataset. Hãy bấm '+ Add Data' upload generate-dataset.zip để train trên cả 2 tập data!")

# Chuẩn hoá cấu trúc nếu bị lồng thư mục
if os.path.exists(f"{synth_dest}/generate-dataset"):
    for item in os.listdir(f"{synth_dest}/generate-dataset"):
        shutil.move(f"{synth_dest}/generate-dataset/{item}", f"{synth_dest}/{item}")
    shutil.rmtree(f"{synth_dest}/generate-dataset")

!rm -rf data/synth/__MACOSX

# ------------------------------------------------------------------------------
# 3. CHẠY PREPARE DỮ LIỆU TASK 3
# ------------------------------------------------------------------------------
print("\n=== 3. Chạy chuẩn bị dữ liệu Task 3 ===")
!python scripts/task3_prepare.py --filter-synthetic strict

print("\nThống kê các phân vùng dữ liệu trong data/processed/task3:")
!ls -lh data/processed/task3


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Đã thiết lập CUDA_VISIBLE_DEVICES = 0.")


In [ ]:
import os, subprocess

adapter_dir = "checkpoints/task3/mixed_balanced"

if MODE == "qwen_qlora":
    print("=" * 70)
    print(">>> BẮT ĐẦU HUẤN LUYỆN QWEN2.5-CODER-7B QLORA TRÊN 2 TẬP DỮ LIỆU")
    print(f"    - Chế độ thí nghiệm: {EXPERIMENT} (Real 26 + Synthetic 496)")
    print(f"    - Số epochs: {EPOCHS}")
    print(f"    - Thư mục lưu checkpoint: {adapter_dir}")
    print("=" * 70)

    train_cmd = [
        "python", "scripts/train_task3.py",
        "--config", "configs/task3/qlora.yaml",
        "--experiment", EXPERIMENT,
        "--output-dir", adapter_dir
    ]
    if MAX_STEPS:
        train_cmd.extend(["--max-steps", str(MAX_STEPS)])

    subprocess.run(train_cmd, check=True)
    print("\n✅ HUẤN LUYỆN HOÀN TẤT! LoRA adapter đã được lưu tại:", adapter_dir)
else:
    print(f"Bỏ qua bước huấn luyện (MODE = '{MODE}').")


In [ ]:
import os, json, subprocess, shutil
from pathlib import Path

output_file = f"outputs/task3/{MODE}_{EVAL_SPLIT}.predictions.json"
os.makedirs("outputs/task3", exist_ok=True)

# 1. Chuẩn bị Taxonomy đầu vào cho tập Public Test (328 mẫu)
taxonomy_file = "outputs/task3/public_baseline_taxonomy.json"
if EVAL_SPLIT == "public":
    from scripts.build_task3_submission import build_default_taxonomy
    if not os.path.exists(taxonomy_file):
        build_default_taxonomy(Path("data/raw/public_test_release"), Path(taxonomy_file))

print(f"\n>>> BẮT ĐẦU CHẠY SUY LUẬN TRÊN TẬP: {EVAL_SPLIT}...")

infer_cmd = ["python", "scripts/run_task3.py", "--split", EVAL_SPLIT, "--output", output_file]

if EVAL_SPLIT == "public":
    infer_cmd.extend(["--taxonomy-source", "task2-json", "--taxonomy-json", taxonomy_file, "--allow-unverified-task2-export"])
else:
    infer_cmd.extend(["--taxonomy-source", "gold"])

if MODE == "gemini":
    infer_cmd.extend(["--config", "configs/task3/gemini.yaml"])
elif MODE == "openai":
    infer_cmd.extend(["--config", "configs/task3/openai.yaml"])
elif MODE == "qwen_baseline":
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml"])
elif MODE == "qwen_qlora":
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml", "--adapter", adapter_dir])

print("Lệnh thực thi:", " ".join(infer_cmd))
subprocess.run(infer_cmd, check=True)
print(f"\n✅ Đã sinh xong kết quả suy luận tại: {output_file}")


In [ ]:
import os, json, shutil, subprocess

traces_file = output_file.replace(".predictions.json", ".predictions.traces.json")
eval_file = output_file.replace(".predictions.json", ".evaluation.json")

# 1. Chạy đánh giá Compliance và Báo cáo kiểm định
eval_cmd = [
    "python", "scripts/evaluate_task3.py",
    "--data-dir", "data/processed/task3",
    "--split", EVAL_SPLIT,
    "--predictions", output_file,
    "--traces", traces_file,
    "--taxonomy-source", ("task2-json" if EVAL_SPLIT == "public" else "gold"),
    "--output", eval_file
]
subprocess.run(eval_cmd, check=True)

# 2. Đóng gói file nộp bài chuẩn tên `task3_submission.json` ra /kaggle/working/
submission_dest = "/kaggle/working/task3_submission.json"
shutil.copy(output_file, submission_dest)
if os.path.exists(eval_file):
    shutil.copy(eval_file, f"/kaggle/working/{os.path.basename(eval_file)}")

print("\n" + "=" * 30 + " KẾT QUẢ ĐÁNH GIÁ COMPLIANCE " + "=" * 30)
with open(eval_file, "r", encoding="utf-8") as f:
    eval_res = json.load(f)
    print(json.dumps(eval_res.get("summary", eval_res), indent=2, ensure_ascii=False))

# 3. Kiểm tra tính hợp lệ của file nộp bài (328 mẫu chuẩn format)
with open(submission_dest, "r", encoding="utf-8") as f:
    preds = json.load(f)

print("\n" + "=" * 30 + " KIỂM TRA FILE NỘP BÀI (SUBMISSION) " + "=" * 30)
print(f"Tổng số mẫu trong file nộp bài: {len(preds)}")
print("3 mẫu đầu tiên:")
print(json.dumps(preds[:3], indent=2, ensure_ascii=False))

print("\nDanh sách các file trong /kaggle/working/ sẵn sàng tải về:")
!ls -lh /kaggle/working
